# USL Championship — Visualizations

Loads pre-built DataFrames from parquet. Run the three template notebooks first to populate `data/`.

In [ ]:
from typing import Final

import numpy as np
import pandas as pd

pd.options.display.max_columns = 999
pd.set_option("expand_frame_repr", False)

In [ ]:
TEAM: Final[str] = "LOU"

In [ ]:
team_stats = pd.read_parquet("../data/team_stats.parquet")
games = pd.read_parquet("../data/games.parquet")
players = pd.read_parquet("../data/players.parquet")
gk_players = pd.read_parquet("../data/gk_players.parquet")

team_games = games[
    (games["home_team"] == TEAM) | (games["away_team"] == TEAM)
].reset_index(drop=True)

In [ ]:
display(team_stats.head())

In [ ]:
display(games.head())

In [ ]:
display(team_games.head())

In [ ]:
display(players.head())

In [ ]:
display(gk_players.head())

## Team View

Flattens the home/away symmetric structure of `team_games` into a single team perspective
driven by `TEAM`. All columns reflect the selected team's side — opponent, score, xG, points,
game number, days rest, and travel — making downstream aggregations team-agnostic.

In [ ]:
_is_home = team_games["home_team"] == TEAM

team_view = team_games.assign(
    # Context
    is_home          = _is_home,
    opponent         = team_games["away_team"].where(_is_home, team_games["home_team"]),
    manager          = team_games["home_manager"].where(_is_home, team_games["away_manager"]),
    opponent_manager = team_games["away_manager"].where(_is_home, team_games["home_manager"]),
    # Venue — always the actual match venue; is_home indicates if it's the team's ground
    venue            = team_games["stadium"],
    venue_city       = team_games["city"],
    venue_state      = team_games["state"],
    # Score and result from the team's perspective
    gf               = team_games["home_score"].where(_is_home, team_games["away_score"]),
    ga               = team_games["away_score"].where(_is_home, team_games["home_score"]),
    goal_diff        = team_games["goal_diff"].where(_is_home, -team_games["goal_diff"]),
    result           = np.select(
                           [team_games["result"] == TEAM, team_games["result"] == "DRAW"],
                           ["W", "D"],
                           default="L",
                       ),
    pts              = team_games["home_pts"].where(_is_home, team_games["away_pts"]),
    season_pts       = team_games["home_season_pts"].where(_is_home, team_games["away_season_pts"]),
    ppg              = team_games["home_ppg"].where(_is_home, team_games["away_ppg"]),
    # xG from the team's perspective
    xgf              = team_games["home_team_xgoals"].where(_is_home, team_games["away_team_xgoals"]),
    xga              = team_games["away_team_xgoals"].where(_is_home, team_games["home_team_xgoals"]),
    xg_diff          = team_games["xg_diff"].where(_is_home, -team_games["xg_diff"]),
    xresult          = np.select(
                           [team_games["xresult"] == TEAM, team_games["xresult"] == "DRAW"],
                           ["W", "D"],
                           default="L",
                       ),
    xpts             = team_games["home_xpoints"].where(_is_home, team_games["away_xpoints"]),
    # Game context
    game_number      = team_games["home_game_number"].where(_is_home, team_games["away_game_number"]),
    days_rest        = team_games["home_days_rest"].where(_is_home, team_games["away_days_rest"]),
    # Travel distance: 0 at home, actual distance when away
    travel_distance  = team_games["away_travel_distance"].where(~_is_home, 0),
)

_col_order = [
    "game_id", "season_name", "matchday", "date", "day_of_week", "time_est", "date_time_est",
    "is_home", "opponent", "manager", "opponent_manager", "referee",
    "venue", "venue_city", "venue_state",
    "gf", "ga", "goal_diff", "result", "pts", "season_pts", "ppg",
    "xgf", "xga", "xg_diff", "xresult", "xpts",
    "expanded_minutes", "attendance", "game_number", "days_rest", "travel_distance",
    "temp_max_f", "temp_min_f", "precip_in", "snowfall_in",
    "status", "knockout_game", "last_updated_utc",
]
team_view = team_view[[c for c in _col_order if c in team_view.columns]].reset_index(drop=True)

print(TEAM)
display(team_view.head())